# Chapter 15: Silent Degradation

A ranker can get worse without anything breaking. This notebook retrains the Chapter 6 XGBoost
ranker on Yambda, then corrupts the frame it scores in three ways that every schema check accepts:

- **zero_fill_all**: the two most important eligible features arrive as 0 for every request
- **zero_fill_segment**: the same, for a 30 percent user segment chosen by hashing the user ID
- **mean_fill_all**: the same two features arrive as their training-set mean

An uncorrupted **clean** control is scored alongside. Every monitor compares the scored test day
with a reference day, the last day before it, which the model never trained or early-stopped on.
The question is which monitors fire, and which stay green, for the aggregate and for the segment.

**Running it.** Set `YAMBDA_DATA_DIR` to the folder that holds Yambda `flat/50m/listens.parquet`
(see `README.md`). To check the code path without the dataset, set `CH15_SYNTHETIC=1`; the synthetic
data only exercises the code, and its numbers say nothing about Yambda. The batch version of the
same experiment is `python run_exercise.py`. No number below is typed in by hand: every table and
plot is computed by the run in the next cells.

In [ ]:
import logging
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Markdown, display

HERE = Path.cwd()
if not (HERE / "run_exercise.py").is_file():
    raise RuntimeError("Start Jupyter from the chapter15_recsysops folder")
sys.path.insert(0, str(HERE))

import corruption as cx
import monitoring_signals as ms
import ranker as rk
import run_exercise as rx

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s", force=True)
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)

SYNTHETIC = os.environ.get("CH15_SYNTHETIC", "0") == "1"
TRAIN_DAYS = int(os.environ.get("CH15_TRAIN_DAYS", "30"))
RUN_NAME = os.environ.get("CH15_RUN_NAME",
                          "smoke_notebook" if SYNTHETIC else f"yambda_{TRAIN_DAYS}d_notebook")
OUTPUT_DIR = os.environ.get("CH15_OUTPUT_DIR", str(HERE / "outputs"))

cfg = rx.ExerciseConfig(ranker=rk.RankerConfig(train_days=TRAIN_DAYS), synthetic=SYNTHETIC,
                        run_name=RUN_NAME, output_dir=OUTPUT_DIR)
cfg

## Step 1: Train the ranker and run every monitor

`run_experiment` loads the last `train_days` of Yambda plus a held-out test day, builds features
with the Chapter 6 Polars pipeline, trains XGBoost with early stopping on a validation window carved
from the training data (never the test day), scores the reference day and the four test-day
scenarios, and writes every result file to `outputs/<run_name>/`.

In [ ]:
result = rx.run_experiment(cfg)
m = result.manifest
print("Rows per window:", m["windows"]["rows"])
print("Best iteration:", m["model"]["best_iteration"], " validation AUC:", round(m["model"]["best_val_auc"], 4))
ch6 = m["provenance"]["chapter6"]
print("Chapter 6 polars_pipeline.py:", ch6["polars_pipeline_sha256"][:12],
      "from commit", ch6.get("source_commit", ch6.get("polars_pipeline_last_commit")),
      " matches recorded source:", ch6.get("body_matches_source"))
print("Data:", m["provenance"]["data"])

## Step 2: Which features get corrupted

The targets are the two features with the highest average gain, among features that are not
identifier encodings, not label-encoded categories, and have more than two distinct values. (For a
binary feature, a mean-fill rounded to the column's integer type would repeat the zero-fill.)

In [ ]:
print("Targets:", result.targets)
print("Fill values:", result.fill_values)
result.eligibility.select("gain_rank", "feature", "gain", "splits", "eligibility", "is_target").head(12)

## Step 3: Zeros and means are schema-valid

The contract below is what a schema validator would enforce: each column keeps its training dtype,
has no nulls or NaNs, and stays inside its natural range (ratios in [0, 1], counts non-negative,
hours 0 to 23). A zero and a training mean satisfy all of it, so the schema monitor has nothing to
catch.

In [ ]:
contract = rk.feature_contract(result.prepared.frames["train"], result.prepared.feature_columns)
rows = []
for name, s in result.scored.items():
    out = ms.schema_pass_rate(s["frame"], contract)
    rows.append({"scenario": name, "schema_pass_rate": out["pass_rate"], "dtypes_unchanged": out["dtype_ok"]})
pl.DataFrame(rows)

## Step 4: Which monitors fire

One status per signal family, scenario, and slice. **FIRES** means the statistic is over its
threshold on the scenario and not on the clean control. **clean also over** means the clean test
day is already over the threshold, so the monitor cannot attribute the change; **FIRES, clean also
over** means it is over on both, but the scenario is worse than clean by more than the threshold.
Thresholds are fixed in `monitoring_signals.Thresholds` before any run.

In [ ]:
order = [("clean", "aggregate"), ("zero_fill_all", "aggregate"), ("zero_fill_segment", "aggregate"),
         ("zero_fill_segment", "segment"), ("zero_fill_segment", "rest"), ("mean_fill_all", "aggregate")]
status = (result.summary
          .with_columns((pl.col("scenario") + " / " + pl.col("slice")).alias("column"))
          .pivot(on="column", index="family", values="status"))
status.select(["family"] + [f"{s} / {c}" for s, c in order])

## Step 5: What the corrupted features look like

Zero-fill moves the whole column to a value most real rows never take. Mean-fill does something
quieter: every row gets the same plausible value, so the column's mean barely moves while its
spread collapses to nothing.

In [ ]:
frames = result.prepared.frames
fig, axes = plt.subplots(1, len(result.targets), figsize=(6 * len(result.targets), 3.8))
for ax, t in zip(np.atleast_1d(axes), result.targets):
    ref = frames["reference"][t].cast(pl.Float64).to_numpy()
    lo, hi = np.quantile(ref, [0.001, 0.999])
    bins = np.linspace(min(lo, 0.0), hi, 50)
    ax.hist(ref, bins=bins, density=True, alpha=0.45, label="reference day")
    ax.hist(result.scored["clean"]["frame"][t].cast(pl.Float64).to_numpy(), bins=bins, density=True,
            histtype="step", linewidth=1.5, label="test day, clean")
    for name, color in (("zero_fill_all", "tab:green"), ("mean_fill_all", "tab:purple")):
        v = result.fill_values["zero" if name.startswith("zero") else "mean"][t]
        ax.axvline(v, color=color, linestyle="--", linewidth=1.5, label=f"{name}: every row = {v:.4g}")
    ax.set_title(t)
    ax.set_xlabel("feature value")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 6: Why mean-fill passes the null, zero, and mean monitors but not KS or PSI

- **Null rate and zero rate** cannot rise: the fill value is neither null nor zero, so both
  fall to zero, and these monitors alarm only on a rise. A feature with many natural zeros on
  the reference day shows a visible drop, which a two-sided monitor would catch.
- **Mean** barely moves: the fill value is the training mean, so the monitor sees only the
  gap between the training mean and the reference-day mean, which stays small unless the served
  feature has drifted away from training.
- **KS** cannot miss it. When every row equals one value m, the current CDF jumps from 0 to 1 at m,
  so the KS statistic is the larger of F_ref(just below m) and 1 - F_ref(m). Those two add up to the
  share of reference rows not equal to m, so KS is at least half that share, and at least 0.5 for a
  feature with no reference mass exactly at m.
- **PSI** cannot miss it either. All current rows land in the one reference-quantile bin that
  holds m, every other bin falls to the floor of 1e-4, and each of those bins adds roughly
  p ln(p / 1e-4) to the index.
- **Quantile shift** fires too, because the 1st and 99th percentiles collapse onto m.

The table shows each of these monitors for the mean-fill scenario, next to the clean control.

In [ ]:
(result.signals
 .filter((pl.col("scenario") == "mean_fill_all") & (pl.col("slice") == "aggregate") & pl.col("is_target"))
 .select("feature", "signal", "statistic", "clean_statistic", "threshold", "status"))

In [ ]:
t = result.targets[0]
ref = np.sort(frames["reference"][t].cast(pl.Float64).to_numpy())
m_fill = result.fill_values["mean"][t]
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(ref, np.arange(1, ref.size + 1) / ref.size, label="reference day")
ax.step([ref.min(), m_fill, m_fill, ref.max()], [0, 0, 1, 1], where="post", label="mean_fill_all")
f_below = np.searchsorted(ref, m_fill, side="left") / ref.size
ax.annotate("", xy=(m_fill, f_below), xytext=(m_fill, 0), arrowprops=dict(arrowstyle="<->"))
ax.text(m_fill, f_below / 2, f"  gap {f_below:.2f}", va="center")
ax.annotate("", xy=(m_fill, 1), xytext=(m_fill, f_below), arrowprops=dict(arrowstyle="<->"))
ax.text(m_fill, (1 + f_below) / 2, f"  gap {1 - f_below:.2f}", va="center")
ax.set_title(f"Empirical CDFs of {t}")
ax.set_xlabel("feature value")
ax.set_ylabel("share of rows at or below")
ax.legend()
plt.tight_layout()
plt.show()

## Step 7: The score distribution

Feature monitors need someone to have chosen the right feature to watch. Score monitors see every
feature at once, through the model.

In [ ]:
bins = np.linspace(0, 1, 41)
fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(result.reference_scores, bins=bins, density=True, alpha=0.4, label="reference day")
for name, s in result.scored.items():
    ax.hist(s["scores"], bins=bins, density=True, histtype="step", linewidth=1.5, label=name)
ax.set_xlabel("predicted completion probability")
ax.set_ylabel("density")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
(result.signals.filter((pl.col("layer") == "score") & (pl.col("slice") == "aggregate"))
 .select("scenario", "signal", "statistic", "threshold", "status"))

## Step 8: Quality and calibration, once labels arrive

Completion labels for the test day arrive after the day ends. AUC and log loss are computed over
all test rows; GAUC and NDCG@10 rank each user's own listens; ECE and the calibration gap compare
predicted probabilities with observed completion rates.

GAUC and NDCG@10 only compare items within one user. A corrupted feature that is constant
across a user's rows, such as the `user_*` lookup features, shifts all of that user's scores
together, so the order within the user changes only through the model's interactions with
other features. These two metrics then move far less than AUC, log loss, and calibration,
which compare scores across users. Item features vary across a user's listens; to corrupt
them instead, run `python run_exercise.py --target_scope item`.

In [ ]:
q = result.quality.filter(pl.col("slice") == "aggregate")
q.pivot(on="scenario", index=["metric", "reference"], values="value")

In [ ]:
rel = result.reliability.filter(pl.col("slice") == "aggregate")
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.plot([0, 1], [0, 1], color="grey", linewidth=1, linestyle=":")
for name in ("reference_day", "clean", "zero_fill_all", "zero_fill_segment", "mean_fill_all"):
    r = rel.filter((pl.col("scenario") == name) & (pl.col("n") > 0))
    ax.plot(r["mean_pred"], r["observed"], marker="o", markersize=3, label=name)
ax.set_xlabel("mean predicted probability (bin)")
ax.set_ylabel("observed completion rate (bin)")
ax.set_title("Reliability, aggregate")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Step 9: A segment can hide inside the aggregate

In **zero_fill_segment** only the hashed 30 percent of users are corrupted. The aggregate mixes
them with untouched traffic, so every statistic is diluted; the **segment** slice shows the damage
at full strength and the **rest** slice shows it does not leak to other users.

In [ ]:
print("Segment share of test-day users:", round(m["segment"]["test_user_share"], 3),
      " of rows:", round(m["segment"]["test_row_share"], 3))
(result.quality.filter(pl.col("scenario") == "zero_fill_segment")
 .pivot(on="slice", index="metric", values="delta_vs_clean"))

In [ ]:
(result.signals
 .filter((pl.col("scenario") == "zero_fill_segment") & pl.col("is_target") & pl.col("signal").is_in(["zero_rate", "ks", "psi"]))
 .pivot(on="slice", index=["feature", "signal"], values="statistic"))

## Step 10: What fires on a clean day

These are the monitors already over threshold on the clean test day, with no corruption at all.
"False alarm" here means only that the injected fill did not cause the alarm. On the full
Yambda data the clean-day alarms have two sources. One account behaves very differently on the
test day than on the reference day, playing far more often and completing almost every play.
It shifts user and item features, the score distribution, AUC, and calibration in every slice
that contains it: the aggregate, and whichever of `segment` or `rest` it hashes into. That is a
real change in the data, and an alert on it deserves a look. Most of the remaining alarms are
quantile shifts on heavy-tailed features, whose extreme percentiles move by more than the
threshold between two ordinary days. A monitor like that pages someone for nothing and trains
them to ignore it. Features listed as exempt are computed but not alerted on, for the reason
given.

In [ ]:
display(rx.clean_false_alarms(result.signals))
for f, why in result.manifest["monitor_exemptions"].items():
    print(f"exempt: {f}: {why}")

## Result files

`outputs/<run_name>/` holds `signal_summary.md` (the tables above), `signal_summary.csv`,
`signals_long.csv` (every monitor, scenario, and slice), `quality_metrics.csv`,
`reliability_bins.csv`, `feature_importance.csv`, `clean_false_alarms.csv`, `run_manifest.json`
(configuration, thresholds, data and Chapter 6 provenance, package versions, timings), and
`model/` (the XGBoost model and the fitted Chapter 6 pipeline).

In [ ]:
print(result.out_dir)
sorted(p.name for p in result.out_dir.iterdir())